# Function Calling 기초

Function Calling은 모델이 애플리케이션에 함수 실행을 요청할 수 있게 하는 기능입니다. Tool Calling이라고도 합니다. 모델은 함수의 이름과 인자를 생성하고, 애플리케이션은 해당 정보를 검증한 뒤 실제 함수를 실행합니다.

이 노트북은 **Responses API의 Function Calling 형식**을 기준으로 설명합니다. Chat Completions API도 같은 개념을 지원하지만 요청·응답 객체의 필드 구조가 다르므로 두 API의 형식을 섞어 사용하면 안 됩니다.

공식 문서: [OpenAI Function Calling 가이드](https://developers.openai.com/api/docs/guides/function-calling)

## 모델과 애플리케이션의 책임

| 구분 | 책임 |
| :--- | :--- |
| 모델 | 사용자 요청을 해석하고 필요한 함수명과 인자를 생성합니다. |
| 애플리케이션 | 함수명과 인자를 확인하고 실제 Python 함수를 실행한 뒤 결과를 모델에 전달합니다. |
| Responses API | 모델 응답과 이전 응답을 연결하고, `previous_response_id`를 이용한 대화 상태 관리를 지원합니다. |

모델은 Python 함수를 직접 실행하지 않습니다. `function_call`은 실행 결과가 아니라 애플리케이션에 보내는 실행 요청입니다.

## 기본 처리 순서

1. 애플리케이션이 함수 정의를 `tools`에 등록합니다.
2. 사용자 입력과 `tools`를 Responses API에 전달합니다.
3. 모델이 일반 텍스트 또는 `function_call` 항목을 반환합니다.
4. 애플리케이션이 `response.output`에서 `function_call`을 찾습니다.
5. 함수명과 인자를 확인한 뒤 실제 Python 함수를 실행합니다.
6. 함수 결과를 `function_call_output` 형식으로 다시 전달합니다.
7. 후속 요청에는 직전 응답의 `response.id`를 `previous_response_id`로 전달합니다.
8. 최종 텍스트는 `response.output_text`에서 읽습니다.

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json
import pytz
from datetime import datetime
from rich.pretty import pprint

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY") 
client = OpenAI(api_key=api_key)

def get_ai_response(input_list, tools=None, previous_response_id=None):
    return client.responses.create(
        model="gpt-5.6-luna",
        input=input_list,
        tools=tools,
        previous_response_id=previous_response_id,  # 변경
    )

## Tool Schema

`tools`에는 모델이 호출할 수 있는 함수 정의를 배열로 전달합니다. Python 함수 객체가 아니라 함수명, 설명, 인자 구조를 표현한 JSON Schema를 전달합니다.

### Responses API 필드 구조

| 경로 | 타입 또는 값 | 설명 |
| :--- | :--- | :--- |
| `tools[].type` | `"function"` | 도구가 사용자 정의 함수임을 나타냅니다. |
| `tools[].name` | `string` | 모델과 애플리케이션이 사용할 함수명입니다. |
| `tools[].description` | `string` | 함수의 용도와 호출 조건입니다. |
| `tools[].parameters` | JSON Schema | 모델이 생성할 함수 인자의 구조입니다. |
| `tools[].strict` | `boolean` | 선택 사항이며, 함수 인자를 스키마에 엄격하게 맞춥니다. |
| `parameters.properties` | `object` | 함수가 받을 수 있는 인자를 정의합니다. |
| `parameters.required` | `array` | 필수 인자 이름을 지정합니다. |
| `parameters.additionalProperties` | `false` | Strict mode에서 정의하지 않은 인자를 금지합니다. |

Chat Completions에서 사용하는 `tools[].function.name` 구조와 달리, Responses API에서는 현재 코드처럼 `name`, `description`, `parameters`를 함수 도구 객체에 직접 작성합니다.

### 모델이 반환하는 함수 정보

Responses API의 함수 호출은 `response.output`에 다음과 같은 항목으로 포함됩니다.

```json
{
  "type": "function_call",
  "name": "get_current_time",
  "arguments": "{\"timezone\":\"Asia/Seoul\"}",
  "call_id": "call_abc123"
}
```

각 필드의 역할은 다음과 같습니다.

| 필드 | 용도 |
| :--- | :--- |
| `type` | `function_call`인지 확인합니다. |
| `name` | 실행할 Python 함수를 선택합니다. |
| `arguments` | JSON 문자열 형태의 함수 인자입니다. |
| `call_id` | 함수 실행 결과를 원래 호출과 연결합니다. |

현재 코드는 `json.loads(function_call.arguments)`로 인자를 변환하고 `function_call.name`이 `get_current_time`인지 확인한 뒤 함수를 실행합니다.

### Strict mode

현재 도구 정의에는 `strict`와 `additionalProperties`가 없으므로 비엄격 방식으로 동작합니다. Strict mode를 사용하려면 다음을 추가해야 합니다.

```json
{
  "strict": true,
  "parameters": {
    "additionalProperties": false
  }
}
```

In [2]:
def get_current_time(timezone: str = "Asia/Seoul"):
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    return f"{now} {timezone}"


tools = [
    {
        "type": "function",
        "name": "get_current_time",
        "description": "해당 타임존의 날짜와 시간을 반환합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "timezone": {
                    "type": "string",
                    "description": "예: Asia/Seoul",
                }
            },
            "required": ["timezone"],
        },
    }
]

## Responses API Function Calling 처리 과정

현재 코드는 `client.responses.create()`와 `previous_response_id`를 사용합니다.

### 1단계: 첫 요청 준비

첫 요청의 `input_list`에는 시스템 메시지와 사용자 메시지가 포함됩니다.

```text
system → user
```

`tools`에는 모델이 호출할 수 있는 함수 정의를 전달합니다. 첫 요청에는 이전 응답이 없으므로 `previous_response_id`는 `None`입니다.

### 2단계: 응답 확인

Responses API의 결과는 `ai_response.output`에 여러 항목으로 포함될 수 있습니다.

```python
function_calls = [
    item
    for item in ai_response.output
    if item.type == "function_call"
]
```

함수 호출이 없으면 `ai_response.output_text`를 최종 답변으로 출력합니다.

### 3단계: 함수 호출 처리

함수 호출이 있으면 다음 값을 사용합니다.

| 값 | 용도 |
| :--- | :--- |
| `function_call.name` | 실행할 함수 선택 |
| `function_call.arguments` | JSON 문자열 형태의 인자 |
| `function_call.call_id` | 함수 요청과 실행 결과 연결 |

현재 예제는 `get_current_time`만 실행하도록 제한합니다.

```python
if function_call.name == "get_current_time":
```

모델이 생성한 인자는 외부 입력으로 취급해야 합니다. 현재 예제는 학습 목적의 최소 구현이므로 JSON 파싱 오류나 잘못된 타임존에 대한 예외 처리는 생략되어 있습니다.

### 4단계: 함수 결과 생성

애플리케이션이 함수를 직접 실행하고 결과를 `function_call_output` 항목으로 만듭니다.

```json
{
  "type": "function_call_output",
  "call_id": "call_abc123",
  "output": "2026-08-30 15:30:00 Asia/Seoul"
}
```

`call_id`는 모델이 반환한 `function_call.call_id`와 같아야 합니다.

### 5단계: 함수 결과 전달

함수 결과를 보낼 때 직전 응답의 ID를 함께 전달합니다.

```python
ai_response = get_ai_response(
    function_outputs,
    tools=tools,
    previous_response_id=ai_response.id,
)
```

전체 대화 기록이나 첫 번째 응답의 `output`을 직접 다시 추가할 필요는 없습니다. Responses API가 `previous_response_id`를 이용해 직전 응답과 후속 요청을 연결합니다.

### 6단계: 다음 사용자 턴 연결

최종 응답 ID를 저장한 뒤 로컬 입력 목록을 비웁니다.

```python
previous_response_id = ai_response.id
input_list.clear()
```

다음 사용자 입력은 저장된 `previous_response_id`를 통해 이전 대화와 연결됩니다. 따라서 애플리케이션은 전체 메시지 목록이 아니라 마지막 응답 ID만 관리합니다.

In [3]:

# 초기 시스템 프롬프트
input_list = [
    {
        "role": "system",
        "content": "너는 사용자를 도와주는 상담사야.",
    }
]

previous_response_id = None

while True:
    user_input = input("사용자\t: ")

    if user_input == "exit":
        break

    input_list.append(
        {
            "role": "user",
            "content": user_input,
        }
    )

    ai_response = get_ai_response(
        input_list,
        tools=tools,
        previous_response_id=previous_response_id,
    )

    pprint(f"[디버깅용] AI 원본 응답: {ai_response}")

    function_calls = [
        item
        for item in ai_response.output
        if item.type == "function_call"
    ]

    if function_calls:
        function_outputs = []

        for function_call in function_calls:
            arguments = json.loads(function_call.arguments)

            if function_call.name == "get_current_time":
                function_outputs.append(
                    {
                        "type": "function_call_output",
                        "call_id": function_call.call_id,
                        "output": get_current_time(
                            timezone=arguments["timezone"]
                        ),
                    }
                )

        ai_response = get_ai_response(
            function_outputs,
            tools=tools,
            previous_response_id=ai_response.id,
        )

    print("AI\t: " + ai_response.output_text)

    previous_response_id = ai_response.id
    input_list.clear()

'[디버깅용] AI 원본 응답: Response(id=\'resp_06141cc89232cabe006a940885a9e487d08dd18ce1bf95ef09\', created_at=1788086405.0, error=None, incomplete_details=None, instructions=None, metadata={}, model=\'gpt-5.6-luna\', object=\'response\', output=[ResponseReasoningItem(id=\'rs_06141cc89232cabe006a9408869bbc87d0a571effe4f3936ce\', summary=[], type=\'reasoning\', content=[], encrypted_content=\'gAAAAABqlAiHMxkJ3rWMhFMqtVrjnYefv2lgEn9taKjXa2eCk7vC3bvXukfetpUlE_GJTPbyCN9B3bDf-nH0trqL_ftLYvVOn8s3RlQBmjVaspTtCCClkUO59zBgsiM7RehyCz-nRXqSfmpUyyPrrWNbcSwdHK2fqR5GY4zlwlO8fJdTh8aRJjrKfOMhDoTYawd9JboK80QDdDK9dyU0lbOyUSI3OAepWRe9uoBqHKO3MKMjKJqBBZM7qiPf2kkPADpJtSK6_d2L2WWHJwsRyCJltZBZdv4ZOV7B1OHQSdPoLVMlSYEdRxhmiqpoSvYiNWnA3BVAVjz1WJJTTCtAZJ1UZcwWAmLVW2MYi8CGI6yyCvqCKZK0Or1ptlbIK0FhzsoUgOgf8K87yWagVSygX_RtiVXj6eSsqoLNTfZ-GRNjueGA2tIKSBPbbrutHijyWYGzwDkACMqueW_b1asC2CuhAmKSooIKBObhP1zrNg2BWPNl6BcfpfF8qEgU7aFgBlfdU4iF4ORzxqZRfKCREZ00UBH7gHgRYs2iYR78oYmGC5Tfi-GyFDGmZaGCQaGflfve4LdNF7y_4OEmlFIAZJ93OR4xXrw7kOKwyZEdmZkLhqgu9HbgHlwu3TcmwEk5ydiWiepjZNrISAYb-jQKxsmOlWGerEdi0aBoDlmdZn65gX1VjNt_sG89x1zNjGaVaiFOvoSyn-WzT5hPpB4EPVHlVyO3VWYsTiEp0oJ3uvSKpRCwNr4IJOwH5846MDXBhbXMhNY7M0S72vvzsPGF3wkd_ae-c4Uarp-mXSNccXoOw4Uy06Uyem2iR9wCnyYt27lcBUhptVAkmkGJkn_aw-nMZ51X2Y43YWvPdUkcwyW8U31ju63gsZ4zHqzZH-wBBMf4yM8LanRtvfekvGlHqy7PxsI7z3d4lS8yQJubhR3B9yCQscUjCkcsFucL14JYNAltn9SfwNQjxPlHFioFLBN81J3z4WX8NRkqKrFk7oKO_pirn6HxOBjMWKGaHjG5cCiEHDL6Ct6Aarf833NWjFmlb8inP_QD1gWugIfPurj4I3d7Kuh0gfs3Yp60IY7wniEzhhPdhlEZ_ZZZq33UWrZ5UW_DfOHCmBWarPaqzm5UtCq8XBuzS07ujp_YrABxA8WH70eOh4Xx1B9jI9288mDbhH7vVsH1VN7C1vnXgongQhVeH91GwAXd8nmSzCA-zCqEsgZcfgbDOMwB5_4dNfh3NTCCNrs_iPda8wNQOCs0g_xg6eewdh8=\', status=None), ResponseFunctionToolCall(arguments=\'{"timezone":"Asia/Seoul"}\', call_id=\'call_syQuONfcsLeZAzr8A6EpaLjr\', name=\'get_current_time\', type=\'function_call\', id=\'fc_06141cc89232cabe006a940886d23087d0ac5577384ab12f85\', caller=None, namespace=None, status=\'completed\')], parallel_tool_calls=True, temperature=1.0, tool_choice=\'auto\', tools=[FunctionTool(name=\'get_current_time\', parameters={\'type\': \'object\', \'properties\': {\'timezone\': {\'type\': \'string\', \'description\': \'예: Asia/Seoul\'}}, \'required\': [\'timezone\'], \'additionalProperties\': False}, strict=True, type=\'function\', allowed_callers=None, defer_loading=None, description=\'해당 타임존의 날짜와 시간을 반환합니다.\', output_schema=None)], top_p=0.98, background=False, completed_at=1788086407.0, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_options=None, prompt_cache_retention=\'24h\', reasoning=Reasoning(context=\'all_turns\', effort=\'medium\', generate_summary=None, mode=\'standard\', summary=None), safety_identifier=None, service_tier=\'default\', status=\'completed\', text=ResponseTextConfig(format=ResponseFormatText(type=\'text\'), verbosity=\'medium\'), top_logprobs=0, truncation=\'disabled\', usage=ResponseUsage(input_tokens=77, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=40, output_tokens_details=OutputTokensDetails(reasoning_tokens=16), total_tokens=117, compute_units=None), user=None, billing={\'payer\': \'developer\'}, frequency_penalty=0.0, presence_penalty=0.0, store=True, tool_usage={\'image_gen\': {\'input_tokens\': 0, \'input_tokens_details\': {\'image_tokens\': 0, \'text_tokens\': 0}, \'output_tokens\': 0, \'output_tokens_details\': {\'image_tokens\': 0, \'text_tokens\': 0}, \'total_tokens\': 0}, \'web_search\': {\'num_requests\': 0}})'

AI	: 한국 표준시(KST)로 현재 **2026년 8월 30일 오후 7시 40분**이에요.
